In [25]:
!pip install PyPDF2

In [26]:
import json
import re
import PyPDF2

In [27]:
def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                text += page.extract_text() + " "
    except FileNotFoundError:
        print(f"Error: File {pdf_path} tidak ditemukan")
    return text

In [28]:
def clean_text(text):
    text = re.sub(r'-\d+-', '', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [29]:
def generate_dataset(raw_text):
    dataset = []

    cleaned_raw = clean_text(raw_text)

    start_match = re.search(r'(Pasal 1\b.*)', cleaned_raw, re.IGNORECASE)
    if not start_match:
        return dataset

    main_content = start_match.group(1)

    pasal_list = re.split(r'(Pasal \d+)', main_content)

    for i in range(1, len(pasal_list)-1, 2):
        nama_pasal = pasal_list[i].strip()
        isi_pasal = pasal_list[i+1].strip()

        if "Ditetapkan di Jakarta" in isi_pasal:
            isi_pasal = isi_pasal.split("Ditetapkan di Jakarta")[0].strip()
        if "Agar setiap orang mengetahuinya" in isi_pasal:
            isi_pasal = isi_pasal.split("Agar setiap orang mengetahuinya")[0].strip()

        data_point = {
            "instruction": f"Jelaskan apa saja ketentuan yang dimuat dalam {nama_pasal} Peraturan Menteri Kesehatan Nomor 10 Tahun 2024!",
            "input": "",
            "output": f"Berdasarkan {nama_pasal}, berikut adalah ketentuannya: {isi_pasal}"
        }
        dataset.append(data_point)

    return dataset

In [30]:
pdf_file_path = "permenkes-no-10-tahun-2024.pdf"
raw_text = extract_text_from_pdf(pdf_file_path)

if raw_text:
    cleaned_data = generate_dataset(raw_text)

    output_file = "dataset_permenkes.jsonl"
    with open(output_file, "w", encoding="utf-8") as f:
        for item in cleaned_data:
            f.write(json.dumps(item) + "\n")

    print(f"Berhasil mengekstrak {len(cleaned_data)} pasal menjadi dataset!\n")

    print("--- DEMO HASIL DATASET ---")
    for i, data in enumerate(cleaned_data[:10]):
        print(f"[{i+1}] Instruction : {data['instruction']}")
        print(f"    Output      : {data['output']}\n")

Berhasil mengekstrak 15 pasal menjadi dataset!

--- DEMO HASIL DATASET ---
[1] Instruction : Jelaskan apa saja ketentuan yang dimuat dalam Pasal 1 Peraturan Menteri Kesehatan Nomor 10 Tahun 2024!
    Output      : Berdasarkan Pasal 1, berikut adalah ketentuannya: Dalam Peraturan Menteri ini yang dimaksud dengan: 1. Jaringan Dokumentasi dan Informasi Hukum Nasional yang selanjutnya disingkat JDIHN adalah wadah pendayagunaan bersama atas dokumen hukum secara tertib, terpadu, dan berkesinambungan, serta merupakan sarana pemberian pelayanan informasi hukum secara lengkap, akurat, mudah, dan cepat . 2. Jaringan Dokumentasi da n Informasi Hukum di lingkungan Kementerian Kesehatan yang selanjutnya disebut JDIH Kemenkes adalah suatu sistem pengelolaan dan pendaya gunaan bersama dokumen hukum dan informasi hukum di bidang kesehatan secara tertib, terpadu dan berkesinambungan serta m erupakan sarana pemberian pelayanan informasi hukum secara lengkap, akurat, mudah dan cepat. 3. Dokumen Hukum ada

In [31]:
!pip install -q accelerate peft bitsandbytes transformers trl datasets

In [70]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [71]:
dataset = load_dataset("json", data_files="dataset_permenkes.jsonl", split="train")

In [72]:
def format_dataset(example):
    example["text"] = f"User: {example['instruction']}\nModel: {example['output']}"
    return example

dataset = dataset.map(format_dataset)

In [73]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [74]:
model.config.torch_dtype = torch.float16

model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

In [75]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [76]:
model = get_peft_model(model, lora_config)

for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [85]:
training_args = SFTConfig(
    output_dir="./hasil_finetuning",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=50,
    logging_steps=10,
    optim="paged_adamw_8bit",
    fp16=False,
    bf16=False,
    dataset_text_field="text",
    max_length=512,
)

In [86]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)

In [87]:
trainer.train()

Step,Training Loss
10,1.783546
20,1.484295
30,1.251752
40,1.089986
50,1.013643


TrainOutput(global_step=50, training_loss=1.3246443939208985, metrics={'train_runtime': 119.2098, 'train_samples_per_second': 3.355, 'train_steps_per_second': 0.419, 'total_flos': 846827334733824.0, 'train_loss': 1.3246443939208985, 'epoch': 25.0})

In [88]:
trainer.model.save_pretrained("lora_adapter_kesehatan")
print(" Adapter berhasil disimpan")

 Adapter berhasil disimpan


In [89]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

In [91]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [92]:
model = PeftModel.from_pretrained(base_model, "lora_adapter_kesehatan")

In [93]:
def tanya_model(pertanyaan):
    prompt = f"User: {pertanyaan}\nModel:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    jawaban_lengkap = tokenizer.decode(outputs[0], skip_special_tokens=True)
    jawaban_final = jawaban_lengkap.split("Model:")[1].strip()
    return jawaban_final

In [94]:
pertanyaan_user = "Jelaskan apa saja ketentuan yang dimuat dalam Pasal 2 Peraturan Menteri Kesehatan Nomor 10 Tahun 2024!"

print("\n" + "="*50)
print(f"INPUT (Pertanyaan User): \n{pertanyaan_user}")
print("-" * 50)
print(f"OUTPUT (Jawaban Model): \n{tanya_model(pertanyaan_user)}")
print("="*50)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



INPUT (Pertanyaan User): 
Jelaskan apa saja ketentuan yang dimuat dalam Pasal 2 Peraturan Menteri Kesehatan Nomor 10 Tahun 2024!
--------------------------------------------------
OUTPUT (Jawaban Model): 
Pasal 2
Kasir: Republik Indonesia
No: 40/2024

PENYELASINAN

1. Tujuan penyelasinan adalah menyelengkan dan mengatasi kegunaan makanan dan minuman yang diberikan kepada pengunjung.

2. Penyelasinan dilakukan dalam berbagai jenis, seperti:
a. Penginjian
b. Pengolahan
c. Pengendalian
d. Pengelolaan
e. Pengembangan
